# 02 — Inference-Time Active Divergence on DDSP

**Active divergence** (Broad et al., 2021) describes techniques that push a trained generative model to produce outputs that intentionally diverge from its training distribution — not through retraining, but through surgical interventions at inference time.

This notebook applies two families of inference-time active divergence to the **trained DDSP violin synthesiser** from Notebook 01. Model weights are **frozen throughout**.

## Pipeline

```
(f0, loudness) ──► [Decoder] ──► harmonic_dist · global_amp · noise_mag
                                         ↓                   ↓
                          [HarmonicOscillator]   [FilteredNoise]
                                         └──── dry ───────────┘
                                                   ↓
                                              [Reverb IR]
                                                   ↓  audio
```

**§2.1** manipulates the *decoder inputs* (f0, loudness) with out-of-distribution trajectories. **§2.2** intercepts the *synthesis outputs* — transforming harmonic frequencies, amplitudes, and reverb IR before rendering.

> **Design note.** All §2.2 demos drive the model with **real violin f0 + loudness trajectories**. Constant pitches break the model's noise/harmonic balance (ratio ≈1:1 instead of ≈1:25), because the GRU has never seen a perfectly static input.


In [ ]:
%matplotlib inline
import sys, pathlib
import numpy as np
import torch
import torchaudio
import matplotlib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

ROOT = pathlib.Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.divergence.inference import (
    load_model, n_frames,
    constant, pitch_glide, microtonal_glide, fast_am, square_gate,
    random_walk, periodic_pitch,
    get_controls, synth_bent,
    mask_harmonics,
    get_ir,
    specshow, play,
)

matplotlib.rcParams['figure.dpi'] = 72
SR  = 16000
HOP = 64

In [ ]:
CKPT     = ROOT / 'models' / 'ddsp_baseline_violin.pt'
CLIP_DIR = ROOT / 'samples' / 'violin'

model = load_model(str(CKPT))
print(f"Model loaded — {sum(p.numel() for p in model.parameters()):,} params (frozen)")

NF_CLIP = 500    # 2 s at 16 kHz / 64-sample hop
T_CLIP  = NF_CLIP * HOP

CLIP_INDICES = [0, 8, 20, 35, 50, 64, 80, 95]
clips = []
for idx in CLIP_INDICES:
    audio, _ = torchaudio.load(str(CLIP_DIR / f'clip_{idx:04d}.wav'))
    audio = audio.squeeze(0)
    feats = torch.load(str(CLIP_DIR / f'clip_{idx:04d}_features.pt'), weights_only=True)
    clips.append({
        'f0':    feats['f0_hz']   [:NF_CLIP].unsqueeze(0),
        'loud':  feats['loudness'][:NF_CLIP].unsqueeze(0),
        'audio': audio[:T_CLIP],
        'name':  f'clip_{idx:04d}',
    })
    f0v = feats['f0_hz'][:NF_CLIP]
    print(f"  clip[{len(clips)-1}]  clip_{idx:04d}  "
          f"f0 {f0v.min():.0f}–{f0v.max():.0f} Hz")

## Baseline — what the trained DDSP violin sounds like


In [ ]:
c = clips[1]
with torch.no_grad():
    ref = model(c['f0'], c['loud'])

fig, axes = plt.subplots(1, 2, figsize=(14, 3))
specshow(c['audio'].unsqueeze(0), SR, 'Original violin', ax=axes[0])
specshow(ref['audio'],             SR, 'DDSP reconstruction', ax=axes[1])
plt.tight_layout(); plt.show()

print("Original:")
play(c['audio'].unsqueeze(0))
print("Reconstruction:")
play(ref['audio'])


---
## 2.1  Out-of-Distribution Input Sampling

The decoder was trained on violin f0 in roughly **196–3136 Hz** with naturalistic loudness envelopes. Feeding it inputs outside this range forces extrapolation — the model's learned priors snap to unfamiliar territory, producing glitches, timbral anomalies, and novel textures without changing any weights.


### 2.1.1  Pitch Transposition Beyond the Training Range

Preserve the real loudness trajectory, multiply the real f0 by a constant factor.


In [ ]:
c = clips[1]
configs = [
    ('× 0.125  (−3 oct)', 0.125),
    ('× 0.25   (−2 oct)', 0.25),
    ('× 1.0  (reference)', 1.0),
    ('× 4.0   (+2 oct)',   4.0),
]
results = [(lbl, synth_bent(model, c['f0'] * mul, c['loud']))
           for lbl, mul in configs]

fig, axes = plt.subplots(1, 4, figsize=(20, 3))
for ax, (lbl, out) in zip(axes, results):
    specshow(out['audio'], SR, lbl, ax=ax)
plt.tight_layout(); plt.show()
# ▶ Use the interactive widget below to audition these examples.


#### Interactive — pitch offset (semitones)


In [ ]:
def demo_ood_pitch(semitones, clip_idx):
    c = clips[clip_idx]
    mul = 2 ** (semitones / 12)
    out = synth_bent(model, c['f0'] * mul, c['loud'])
    fig, axes = plt.subplots(1, 2, figsize=(14, 3))
    specshow(c['audio'].unsqueeze(0), SR, 'Original', ax=axes[0])
    specshow(out['audio'], SR, f'f0 shifted {semitones:+d} st (×{mul:.2f})', ax=axes[1])
    plt.tight_layout(); display(fig); plt.close(fig)
    play(out['audio'])

interact(demo_ood_pitch,
    semitones=widgets.IntSlider(min=-36, max=36, step=1, value=-24,
                                 description='Semitones', continuous_update=False),
    clip_idx=widgets.Dropdown(options=list(range(len(clips))),
                               value=1, description='Clip'))


### 2.1.2  Unphysical Loudness Envelopes

Keep the real f0 melody; replace loudness with a sinusoidal AM envelope no player can produce. The GRU's recurrent state receives an impossible articulation pattern and produces distorted synth parameters as it tries to make sense of it.


In [ ]:
c = clips[1]
am_demos = [
    ('AM 1 Hz',   fast_am(NF_CLIP, rate_hz=1.0,  depth_db=30.0, base_db=-30.0)),
    ('AM 8 Hz',   fast_am(NF_CLIP, rate_hz=8.0,  depth_db=30.0, base_db=-30.0)),
    ('AM 20 Hz',  fast_am(NF_CLIP, rate_hz=20.0, depth_db=30.0, base_db=-30.0)),
]
am_results = [(lbl, synth_bent(model, c['f0'], ld)) for lbl, ld in am_demos]

fig, axes = plt.subplots(1, 3, figsize=(18, 3))
for ax, (lbl, out) in zip(axes, am_results):
    specshow(out['audio'], SR, lbl, ax=ax)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 3))
for ax, ((lbl, _), (_, ld)) in zip(axes, zip(am_results, am_demos)):
    ax.plot(ld.squeeze().numpy(), linewidth=0.7)
    ax.set(title=lbl + ' — loudness (dB)', xlabel='Frame', ylabel='dB', ylim=(-80, 0))
plt.tight_layout(); plt.show()
# ▶ Use the interactive widget below to audition these examples.


#### Interactive — AM rate and depth


In [ ]:
def demo_am(rate_hz, depth_db, clip_idx):
    c = clips[clip_idx]
    ld = fast_am(NF_CLIP, rate_hz=rate_hz, depth_db=depth_db, base_db=-30.0)
    out = synth_bent(model, c['f0'], ld)
    fig, axes = plt.subplots(1, 2, figsize=(14, 3))
    specshow(out['audio'], SR,
             f'AM  {rate_hz:.0f} Hz  depth={depth_db:.0f} dB', ax=axes[0])
    axes[1].plot(ld.squeeze().numpy(), linewidth=0.7)
    axes[1].set(title='Loudness envelope', xlabel='Frame', ylabel='dB', ylim=(-80, 0))
    plt.tight_layout(); display(fig); plt.close(fig)
    play(out['audio'])

interact(demo_am,
    rate_hz=widgets.FloatSlider(min=0.5, max=40.0, step=0.5, value=8.0,
                                 description='Rate (Hz)', continuous_update=False),
    depth_db=widgets.FloatSlider(min=5.0, max=60.0, step=5.0, value=30.0,
                                  description='Depth (dB)', continuous_update=False),
    clip_idx=widgets.Dropdown(options=list(range(len(clips))),
                               value=1, description='Clip'))


### 2.1.3  Synthetic Pitch Trajectories

Using real loudness as a baseline, we replace f0 with fully synthetic shapes.


In [ ]:
c = clips[2]
torch.manual_seed(7)
f0_rw  = random_walk(NF_CLIP, start=440.0, step_std=20.0, lo=50.0, hi=3000.0)
f0_per = periodic_pitch(440.0, NF_CLIP, rate_hz=0.5, semitone_range=48.0)
f0_sub = microtonal_glide(40.0, semitone_range=2.0, nf=NF_CLIP, cycles=3.0)

traj_configs = [
    ('Brownian walk 50–3000 Hz', f0_rw),
    ('Periodic ±24 st sweep',    f0_per),
    ('Sub-bass micro ±1 st',     f0_sub),
]
traj_results = [(lbl, synth_bent(model, f0, c['loud'])) for lbl, f0 in traj_configs]

fig, axes = plt.subplots(1, 3, figsize=(18, 3))
for ax, (lbl, out) in zip(axes, traj_results):
    specshow(out['audio'], SR, lbl, ax=ax)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 3))
for ax, ((lbl, _), (_, f0)) in zip(axes, zip(traj_results, traj_configs)):
    ax.plot(f0.squeeze().numpy(), linewidth=0.7)
    ax.set(title=lbl + ' — f0 (Hz)', xlabel='Frame', ylabel='Hz')
plt.tight_layout(); plt.show()

for lbl, out in traj_results:
    print(f"  ▶ {lbl}")
    play(out['audio'])


---
## 2.2  Synthesis Chain Hacking

All §2.2 demos drive the decoder with **real violin f0 + loudness** from a clip, then intercept the synthesis chain before rendering.

| Point | Tensor shape | Technique |
|-------|-------------|-----------|
| Base waveform | per-sample, per-harmonic | wavefold, FM |
| Harmonic frequencies | (B, T, K) Hz | inharmonicity |
| Harmonic amplitudes | (B, N, 100) | masking |
| Component levels | scalar gains | harmonic / noise gain |
| Reverb IR | (64000,) | time-reverse |


### 2.2.1  Waveshaping — Wavefold

Each harmonic computes `A_k · global_amp · sin(phase_k)`. **Wavefolding** wraps the sine amplitude back on itself when it exceeds a threshold — equivalent to folding a piece of paper: once a value reaches the edge, it folds back in the other direction.

```
wavefold(p, amount) = |((amount · sin(p) + 1) mod 2) − 1| · 2 − 1
```

At low amount the output is nearly sinusoidal. As amount increases, more folds appear, adding rich harmonic content. High amounts produce very dense, almost noise-like spectra.


In [ ]:
c = clips[0]
fold_demos = [
    ('sin (reference)', 0.0),
    ('wavefold ×2',     2.0),
    ('wavefold ×4',     4.0),
    ('wavefold ×7',     7.0),
]
wave_results = [(lbl, synth_bent(model, c['f0'], c['loud'], wavefold=amt))
                for lbl, amt in fold_demos]

fig, axes = plt.subplots(1, 4, figsize=(20, 3))
for ax, (lbl, out) in zip(axes, wave_results):
    specshow(out['audio'], SR, lbl, ax=ax)
plt.tight_layout(); plt.show()
# ▶ Use the interactive widget below to audition these examples.


#### Interactive — fold amount


In [ ]:
def demo_wavefold(amount, clip_idx):
    c = clips[clip_idx]
    out = synth_bent(model, c['f0'], c['loud'], wavefold=amount)
    fig, axes = plt.subplots(1, 2, figsize=(14, 3))
    specshow(c['audio'].unsqueeze(0), SR, 'Original', ax=axes[0])
    specshow(out['audio'], SR, f'wavefold ×{amount:.1f}', ax=axes[1])
    plt.tight_layout(); display(fig); plt.close(fig)
    play(out['audio'])

interact(demo_wavefold,
    amount=widgets.FloatSlider(min=0.0, max=10.0, step=0.5, value=1.0,
                                description='Amount', continuous_update=False),
    clip_idx=widgets.Dropdown(options=list(range(len(clips))),
                               value=1, description='Clip'))


### 2.2.1b  Frequency Modulation

Adds a sinusoidal phase deviation to each harmonic's carrier:

```
phase_bent_k = ∫ f_k dt  +  β · sin(∫ f_k · ratio dt)
```

Integer ratios keep sidebands harmonic; irrational ratios (√2 ≈ 1.414, φ ≈ 1.618) create bell/gong inharmonic spectra.


In [ ]:
c = clips[3]
fm_demos = [
    ('no FM (ref)',        0.0, 1.0),
    ('β=3  ratio=1',       3.0, 1.0),
    ('β=3  ratio=√2',      3.0, 1.414),
    ('β=6  ratio=φ',       6.0, 1.618),
]
fm_results = [(lbl, synth_bent(model, c['f0'], c['loud'],
                                fm_depth=d, fm_ratio=r))
              for lbl, d, r in fm_demos]

fig, axes = plt.subplots(1, 4, figsize=(20, 3))
for ax, (lbl, out) in zip(axes, fm_results):
    specshow(out['audio'], SR, lbl, ax=ax)
plt.tight_layout(); plt.show()
# ▶ Use the interactive widget below to audition these examples.


#### Interactive — FM depth and ratio


In [ ]:
def demo_fm(beta, ratio, clip_idx):
    c = clips[clip_idx]
    out = synth_bent(model, c['f0'], c['loud'], fm_depth=beta, fm_ratio=ratio)
    fig, axes = plt.subplots(1, 2, figsize=(14, 3))
    specshow(c['audio'].unsqueeze(0), SR, 'Original', ax=axes[0])
    specshow(out['audio'], SR, f'FM  β={beta:.1f}  ratio={ratio:.3f}', ax=axes[1])
    plt.tight_layout(); display(fig); plt.close(fig)
    play(out['audio'])

interact(demo_fm,
    beta=widgets.FloatSlider(min=0.0, max=12.0, step=0.5, value=3.0,
                              description='β (depth)', continuous_update=False),
    ratio=widgets.FloatSlider(min=0.1, max=4.0, step=0.05, value=1.414,
                               description='ratio', continuous_update=False),
    clip_idx=widgets.Dropdown(options=list(range(len(clips))),
                               value=3, description='Clip'))


### 2.2.2  Inharmonicity

In a perfectly harmonic instrument every partial sits exactly at an integer multiple of f0: f_k = k · f0. Real instruments deviate — piano strings are slightly stiff, so higher partials are stretched upward.

We apply the same formula here:

```
f_k_bent = k · f0 · √(1 + B · k²)
```

With **B = 0** the synthesiser is perfectly harmonic. As B increases, higher harmonics are increasingly stretched above their ideal positions, creating bell-like or metallic inharmonicity.

The **h_n** parameter limits which harmonics are affected. Since a violin tone only has ~10–15 active harmonics, keeping **h_n ≈ 12** focuses the detuning on the energy-carrying partials; harmonics above h_n stay harmonic.


In [ ]:
c = clips[4]
inh_demos = [
    ('B = 0  (harmonic)',  0.0),
    ('B = 0.003',          0.003),
    ('B = 0.01',           0.01),
    ('B = 0.04',           0.04),
]
inh_results = [(lbl, synth_bent(model, c['f0'], c['loud'],
                                h_inharmonicity=B, h_n=12))
               for lbl, B in inh_demos]

fig, axes = plt.subplots(1, 4, figsize=(20, 3))
for ax, (lbl, out) in zip(axes, inh_results):
    specshow(out['audio'], SR, lbl, ax=ax)
plt.tight_layout(); plt.show()
# ▶ Use the interactive widget below to audition these examples.


#### Interactive — inharmonicity coefficient and harmonic scope


In [ ]:
def demo_inharmonicity(B, h_n, clip_idx):
    c = clips[clip_idx]
    out = synth_bent(model, c['f0'], c['loud'], h_inharmonicity=B, h_n=h_n)

    # Show how harmonic frequencies shift for the last frame
    f0_val = c['f0'].squeeze()[-1].item()
    k = np.arange(1, h_n + 1)
    f_harm = k * f0_val
    f_bent = k * f0_val * np.sqrt(np.maximum(1 + B * k**2, 1e-4))

    fig, axes = plt.subplots(1, 3, figsize=(18, 3))
    specshow(c['audio'].unsqueeze(0), SR, 'Original', ax=axes[0])
    specshow(out['audio'], SR, f'B={B:.4f}  h_n={h_n}', ax=axes[1])
    axes[2].vlines(f_harm, 0, 1, colors='steelblue', alpha=0.6, label='harmonic')
    axes[2].vlines(f_bent, 0, 1, colors='tomato',    alpha=0.8,
                   linestyles='--', label='inharmonic')
    axes[2].set(title=f'Harmonic vs bent freqs (h_n={h_n}, f0={f0_val:.0f} Hz)',
                xlabel='Frequency (Hz)', xlim=(0, min(f_bent[-1]*1.2, SR/2)),
                yticks=[])
    axes[2].legend(fontsize=8)
    plt.tight_layout(); display(fig); plt.close(fig)
    play(out['audio'])

interact(demo_inharmonicity,
    B=widgets.FloatSlider(min=0.0, max=0.05, step=0.001, value=0.005,
                          description='B (inharmonicity)', continuous_update=False,
                          style={'description_width': 'initial'}),
    h_n=widgets.IntSlider(min=1, max=40, step=1, value=12,
                          description='h_n', continuous_update=False),
    clip_idx=widgets.Dropdown(options=list(range(len(clips))),
                               value=4, description='Clip'))


### 2.2.3  Component Limiting — Harmonic Masks

Zero out subsets of harmonics to force the model into a spectral sub-space. With `h_n = 12`, the mask operates only on the active harmonic range; harmonics 13–100 (which are near-zero for violin) are unchanged.

| Mask | Acoustic effect |
|------|----------------|
| odd only | Clarinet-like: cylindrical-bore hollow tone |
| even only | Octave-shifted shimmer |
| first K | Low-pass on the harmonic series |
| above K | Remove fundamental: airy, pitched noise |


In [ ]:
c = clips[5]
mask_demos = [
    ('reference',      'all',     10),
    ('odd harmonics',  'odd',      0),
    ('first 6',        'first_k',  6),
    ('above 6',        'above_k',  6),
]
mask_results = [(lbl, synth_bent(model, c['f0'], c['loud'],
                                  h_mask=mode, h_mask_k=k, h_n=12))
                for lbl, mode, k in mask_demos]

fig, axes = plt.subplots(1, 4, figsize=(20, 3))
for ax, (lbl, out) in zip(axes, mask_results):
    specshow(out['audio'], SR, lbl, ax=ax)
plt.tight_layout(); plt.show()
# ▶ Use the interactive widget below to audition these examples.


#### Interactive — mask mode, k, and harmonic scope


In [ ]:
def demo_mask(mode, k, h_n, clip_idx):
    c = clips[clip_idx]
    out  = synth_bent(model, c['f0'], c['loud'], h_mask=mode, h_mask_k=k, h_n=h_n)
    ctrl = get_controls(model, c['f0'], c['loud'])
    orig_d = ctrl['harmonic_dist'].squeeze()[-1].numpy()
    bent_d = mask_harmonics(ctrl['harmonic_dist'], mode, k,
                             n_harmonics=h_n).squeeze()[-1].numpy()

    fig, axes = plt.subplots(1, 3, figsize=(18, 3))
    specshow(c['audio'].unsqueeze(0), SR, 'Original', ax=axes[0])
    specshow(out['audio'], SR, f'Mask: {mode}  k={k}  h_n={h_n}', ax=axes[1])
    axes[2].bar(range(len(orig_d)), orig_d, alpha=0.35, label='original', width=1)
    axes[2].bar(range(len(bent_d)), bent_d, alpha=0.8,  label='masked',   width=1)
    axes[2].axvline(h_n - 0.5, color='r', linestyle='--', linewidth=0.8,
                    label=f'h_n={h_n}')
    axes[2].set(title='Harmonic amplitudes (last frame)', xlabel='Harmonic #', xlim=(0, 40))
    axes[2].legend(fontsize=8)
    plt.tight_layout(); display(fig); plt.close(fig)
    play(out['audio'])

interact(demo_mask,
    mode=widgets.Dropdown(
        options=['all', 'odd', 'even', 'first_k', 'above_k', 'random'],
        value='odd', description='Mode'),
    k=widgets.IntSlider(min=1, max=30, step=1, value=6,
                        description='k', continuous_update=False),
    h_n=widgets.IntSlider(min=1, max=40, step=1, value=12,
                          description='h_n', continuous_update=False),
    clip_idx=widgets.Dropdown(options=list(range(len(clips))),
                               value=5, description='Clip'))


### 2.2.4  Component Level Control

The dry signal is `harmonic_gain × harmonic_audio + noise_gain × noise_audio`. Setting either gain to 0 silences that component entirely — letting you hear harmonics or noise in isolation, or blend them in any proportion.


In [ ]:
def demo_mix(harmonic_gain, noise_gain, clip_idx):
    c = clips[clip_idx]
    out = synth_bent(model, c['f0'], c['loud'],
                     harmonic_gain=harmonic_gain, noise_gain=noise_gain)
    fig, axes = plt.subplots(1, 2, figsize=(14, 3))
    specshow(c['audio'].unsqueeze(0), SR, 'Original', ax=axes[0])
    specshow(out['audio'], SR,
             f'h={harmonic_gain:.1f}  n={noise_gain:.1f}', ax=axes[1])
    plt.tight_layout(); display(fig); plt.close(fig)
    play(out['audio'])

interact(demo_mix,
    harmonic_gain=widgets.FloatSlider(min=0.0, max=2.0, step=0.1, value=1.0,
                                      description='Harmonics', continuous_update=False),
    noise_gain=widgets.FloatSlider(min=0.0, max=2.0, step=0.1, value=1.0,
                                   description='Noise', continuous_update=False),
    clip_idx=widgets.Dropdown(options=list(range(len(clips))),
                               value=0, description='Clip'))


### 2.2.5  Reverse Reverb

The `Reverb` module convolves dry audio with a **learned impulse response** (64 000 samples = 4 s) encoding the training room's acoustic character.

**Time-reversing the IR** produces the classic *reverse reverb* effect: each note is pre-announced by a swelling tail that arrives before the attack. 

In [ ]:
c = clips[7]
out_fwd = synth_bent(model, c['f0'], c['loud'])
out_rev = synth_bent(model, c['f0'], c['loud'], ir_reverse=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 3))
specshow(out_fwd['audio'], SR, 'Normal reverb', ax=axes[0])
specshow(out_rev['audio'], SR, 'Reverse reverb', ax=axes[1])
plt.tight_layout(); plt.show()
# ▶ Use the interactive widget below to audition these examples.


#### Interactive — reverse reverb on/off


In [ ]:
def demo_reverb(reverse, clip_idx):
    c = clips[clip_idx]
    out = synth_bent(model, c['f0'], c['loud'], ir_reverse=reverse)
    fig, axes = plt.subplots(1, 2, figsize=(14, 3))
    specshow(c['audio'].unsqueeze(0), SR, 'Original', ax=axes[0])
    lbl = 'Reverse reverb' if reverse else 'Normal reverb'
    specshow(out['audio'], SR, lbl, ax=axes[1])
    plt.tight_layout(); display(fig); plt.close(fig)
    play(out['audio'])

interact(demo_reverb,
    reverse=widgets.Checkbox(value=True, description='Reverse reverb'),
    clip_idx=widgets.Dropdown(options=list(range(len(clips))),
                               value=7, description='Clip'))


---
## Stacking — Combinatorial Divergence

All interception points compose freely. Three examples:


In [ ]:
# A: Sub-bass + wavefold + inharmonicity within first 12 harmonics
cA = clips[0]
outA = synth_bent(model, cA['f0'] * 0.125, cA['loud'],
                  wavefold=3.0, h_inharmonicity=0.02, h_n=12)

# B: FM (metallic ratio) + reverse reverb
cB = clips[3]
outB = synth_bent(model, cB['f0'], cB['loud'],
                  fm_depth=4.0, fm_ratio=1.414, ir_reverse=True)

# C: Odd harmonics only (h_n=12) + square gate loudness + noise silenced
cC = clips[5]
ldC = square_gate(NF_CLIP, rate_hz=4.0, on_db=-18.0, off_db=-70.0)
outC = synth_bent(model, cC['f0'], ldC,
                  h_mask='odd', h_n=12, noise_gain=0.0)

fig, axes = plt.subplots(1, 3, figsize=(18, 3))
specshow(outA['audio'], SR, 'A: sub-bass + wavefold×3 + inharmonicity', ax=axes[0])
specshow(outB['audio'], SR, 'B: FM (√2) + reverse reverb',              ax=axes[1])
specshow(outC['audio'], SR, 'C: odd harmonics + sq. gate',               ax=axes[2])
plt.tight_layout(); plt.show()

print("A: sub-bass + wavefold×3 + inharmonicity")
play(outA['audio'])
print("B: FM (√2) + reverse reverb")
play(outB['audio'])
print("C: C: odd harmonics + sq. gate")
play(outC['audio'])
